In [2]:
import pandas as pd
import numpy as np

DATA_PATH = "C:\\AMDARI PROJECTS\\NEXYGEN\\nexygen-emissions-forecasting\\data\\raw\\ESG_Data.csv" 

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["Date"],
)

df.dtypes


Date                                datetime64[us]
Year                                         int64
Asset_ID                                       str
Asset_Type                                     str
Location                                       str
Operational_Status                             str
Energy_Type                                    str
Consumption_Units                          float64
Emission_Type                                  str
Emissions_tCO2e                            float64
Target_Emissions_tCO2e                     float64
Reduction_Percentage_vs_BaseYear           float64
dtype: object

In [3]:
#Standardise text fields (lowercase + strip + nice labels)
# Helper: strip spaces
for col in ["Asset_ID", "Asset_Type", "Location", "Operational_Status",
            "Energy_Type", "Emission_Type"]:
    df[col] = df[col].astype(str).str.strip()

In [4]:
df

,Date,Year,Asset_ID,Asset_Type,Location,Operational_Status,Energy_Type,Consumption_Units,Emission_Type,Emissions_tCO2e,Target_Emissions_tCO2e,Reduction_Percentage_vs_BaseYear
0,2020-01-01,2020,A001,ServiceHub,England,Active,ELECTRICITY_KWH,2357.044,Scope 1,0.645377,15004.345,0.00
1,2020-01-01,2020,A001,ServiceHub,England,Active,ELECTRICITY_KWH,2357.044,Scope 2,0.530871,15004.345,0.00
2,2020-01-02,2020,A001,ServiceHub,England,Active,ELECTRICITY_KWH,2440.574,Scope 1,0.688000,15004.345,0.00
3,2020-01-02,2020,A001,ServiceHub,England,Active,ELECTRICITY_KWH,2440.574,Scope 2,0.572372,15004.345,0.00
4,2020-01-03,2020,A001,ServiceHub,England,Active,ELECTRICITY_KWH,2396.825,Scope 1,0.715998,15004.345,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...
236731,2025-12-29,2025,A018,Depot,England,Active,DIESEL_L,220.770,Scope 2,0.140224,11307.747,24.64
236732,2025-12-30,2025,A018,Depot,England,Active,DIESEL_L,186.620,Scope 1,0.699935,11307.747,24.64
236733,2025-12-30,2025,A018,Depot,England,Active,DIESEL_L,186.620,Scope 2,0.149179,11307.747,24.64
236734,2025-12-31,2025,A018,Depot,England,Active,DIESEL_L,155.839,Scope 1,0.593795,11307.747,24.64


In [7]:
#Harmonise to consistent labels:
# Emission_Type -> "Scope 1", "Scope 2"
df["Emission_Type"] = (
    df["Emission_Type"]
    .str.lower()
    .replace({
        "scope1": "scope 1",
        "scope_1": "scope 1",
        "s1": "scope 1",
        "scope2": "scope 2",
        "scope_2": "scope 2",
        "s2": "scope 2",
    })
    .str.title()  # "Scope 1", "Scope 2"
)

# Energy_Type standard names
df["Energy_Type"] = (
    df["Energy_Type"]
    .str.lower()
    .replace({
        "electric": "electricity_kwh",
        "electricity": "electricity_kwh",
        "elec": "electricity_kwh",
        "gas": "gas_kwh",
        "diesel": "diesel_l",
    })
    .str.upper()
)


In [8]:
df["Energy_Type"].value_counts()

Energy_Type
ELECTRICITY_KWH    78912
GAS_KWH            78912
DIESEL_L           78912
Name: count, dtype: int64

In [9]:
#Handle missing and invalid data
# Check for missing values
df.isnull().sum()


Date                                0
Year                                0
Asset_ID                            0
Asset_Type                          0
Location                            0
Operational_Status                  0
Energy_Type                         0
Consumption_Units                   0
Emission_Type                       0
Emissions_tCO2e                     0
Target_Emissions_tCO2e              0
Reduction_Percentage_vs_BaseYear    0
dtype: int64

In [10]:
#Running negative‑value check
(df["Consumption_Units"] < 0).sum(), (df["Emissions_tCO2e"] < 0).sum()


(np.int64(0), np.int64(0))

In [11]:
# Aggregate to daily totals per scope
daily_scope = (
    df.groupby(["Date", "Emission_Type"], as_index=False)
      .agg(
          Emissions_tCO2e=("Emissions_tCO2e", "sum"),
          Consumption_Units=("Consumption_Units", "sum")
      )
)

daily_scope.head()
daily_scope.describe()


,Date,Emissions_tCO2e,Consumption_Units
count,4384,4384.000000,4384.000000
mean,2022-12-31 12:00:00,54.629118,160649.190851
min,2020-01-01 00:00:00,29.885139,121984.004000
25%,2021-07-01 18:00:00,48.414015,151845.102750
50%,2022-12-31 12:00:00,56.057875,161019.140000
75%,2024-07-01 06:00:00,61.323604,170454.563500
max,2025-12-31 00:00:00,73.357614,191731.964000
std,NaN,8.915643,12702.664847


In [23]:
#Add outlier flag and capped emissions
from scipy import stats

def flag_outliers(group, col, z_thresh=3.0):
    z = np.abs(stats.zscore(group[col], nan_policy="omit"))
    return pd.Series(z > z_thresh, index=group.index)

# Flag outliers per scope
daily_scope["is_outlier"] = (
    daily_scope.groupby("Emission_Type", group_keys=False)
               .apply(flag_outliers, col="Emissions_tCO2e")
)

daily_scope["is_outlier"].value_counts()


is_outlier
False    4384
Name: count, dtype: int64

In [13]:
daily_scope.groupby("Emission_Type")["Emissions_tCO2e"].sum()


Emission_Type
Scope 1    131129.696790
Scope 2    108364.355412
Name: Emissions_tCO2e, dtype: float64

In [ ]:
#Capped emissions column
def cap_outliers(group, col, z_thresh=3.0):
    z = np.abs(stats.zscore(group[col], nan_policy="omit"))
    capped = group[col].copy()
    cap_value = group[col].quantile(0.99)
    capped[z > z_thresh] = cap_value
    return capped

daily_scope["Emissions_capped"] = (
    daily_scope.groupby("Emission_Type", group_keys=False)
               .apply(cap_outliers, col="Emissions_tCO2e")
)


In [25]:
daily_scope[["Emissions_tCO2e", "Emissions_capped"]].head()


,Emissions_tCO2e,Emissions_capped
0,54.483339,54.483339
1,65.017719,65.017719
2,59.236587,59.236587
3,64.950840,64.950840
4,59.415912,59.415912


In [26]:
#Add Year and Month for easier grouping
daily_scope["Year"] = daily_scope["Date"].dt.year
daily_scope["Month"] = daily_scope["Date"].dt.to_period("M").astype(str)
daily_scope.head()


,Date,Emission_Type,Emissions_tCO2e,Consumption_Units,is_outlier,Emissions_capped,Year,Month
0,2020-01-01,Scope 1,54.483339,161731.731,False,54.483339,2020,2020-01
1,2020-01-01,Scope 2,65.017719,161731.731,False,65.017719,2020,2020-01
2,2020-01-02,Scope 1,59.236587,165297.418,False,59.236587,2020,2020-01
3,2020-01-02,Scope 2,64.950840,165297.418,False,64.950840,2020,2020-01
4,2020-01-03,Scope 1,59.415912,161509.300,False,59.415912,2020,2020-01


In [27]:
#Save gold daily dataset
GOLD_DAILY_PATH = "../data/processed/gold_daily_scope.csv"
daily_scope.to_csv(GOLD_DAILY_PATH, index=False)
GOLD_DAILY_PATH


'../data/processed/gold_daily_scope.csv'